# Більші моделі

In [1]:
!pip install -q pytorch-lightning wandb tokenizers

In [2]:
import os, re, json, pickle, string, random
from typing import List
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

SEED     = 21
MAX_LEN  = 256
PAD, UNK = 0, 1
IGNORE   = -100
N_TRAIN  = 150_000
EPOCHS   = 3
BATCH    = 128

USE_WANDB     = True
WANDB_PROJECT = 'dl-3'
WANDB_ENTITY  = 'yrachynska-kyiv-school-of-economics'
WANDB_SECRET  = 'WANDB_API_KEY'

pl.seed_everything(SEED)
files = [os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs]
def find(name):
    found = [f for f in files if os.path.basename(f) == name]
    if not found:
        raise FileNotFoundError(f'{name} не знайдено в /kaggle/input. Додайте вихід ноутбука 03 як Input '
                                f'(і зробіть там Save Version). Зараз доступні файли: {[os.path.basename(f) for f in files][:20]}')
    return found[0]

if USE_WANDB:
    try:
        import wandb
        from kaggle_secrets import UserSecretsClient
        wandb.login(key=UserSecretsClient().get_secret(WANDB_SECRET))
        list(wandb.Api().projects(WANDB_ENTITY))
        print(f'W&B: підключено, проєкт {WANDB_ENTITY}/{WANDB_PROJECT}')
    except Exception as e:
        print('W&B НЕ підключено:', e, '\nПеревірте назву секрету, WANDB_ENTITY (адреса проєкту wandb.ai/<entity>/dl-3) та Internet on. Далі логуємо лише в CSV.')
        USE_WANDB = False

print('GPU:', torch.cuda.is_available())

Seed set to 21
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yrachynska (kaggle-casmi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B: підключено, проєкт yrachynska-kyiv-school-of-economics/dl-3
GPU: True


In [3]:
with open(find('prepared.pkl'), 'rb') as f:
    prepared = pickle.load(f)

val_text, lab_text, test_text = prepared['val_text'], prepared['lab_text'], prepared['test_text']
print({k: len(prepared['word'][k]['ids']) for k in ['big', 'val', 'lab', 'test']})
print('словник word:', len(prepared['word']['vocab']), '| словник bpe:', prepared['bpe']['vocab_size'])

{'big': 373790, 'val': 2000, 'lab': 26, 'test': 477}
словник word: 179708 | словник bpe: 16000


In [4]:
def hard_processing(text):
    text = re.sub(r"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\b\w\b\s?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def comp_metric(y_true: List[List[str]], y_pred: List[List[str]]):
    assert len(y_true) == len(y_pred)
    tp, fp, fn, p = 0.0, 0.0, 0.0, 0.0
    for t, pr in zip(y_true, y_pred):
        t = set([hard_processing(s) for s in t])
        pr = set([hard_processing(s) for s in pr])
        tp += len(t & pr); fp += len(pr - t); fn += len(t - pr); p += len(t)
    if tp + fp == 0:
        precision = 1.0 if p == 0 else 0.0
    else:
        precision = tp / (tp + fp)
    if tp + fn == 0:
        recall = 1.0 if p == 0 else 0.0
    else:
        recall = tp / (tp + fn)
    f1 = 0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return round(f1, 8)

EMOJI = re.compile('[\U0001F000-\U0001FFFF\u2600-\u27BF\uFE0F]')
POST_RULES = set()

def clean_span(s, rules):
    if 'newline' in rules:
        s = s.split('\n')[0]
    if 'emoji' in rules:
        s = EMOJI.sub('', s)
    s = s.strip()
    if 'strip' in rules:
        s = s.strip(' \t,.;:!?-–—"«»“”()')
    if 'letters' in rules and not re.search(r'[^\W\d_]', s):
        return None
    if 'upper' in rules and not any(w[0].isupper() for w in re.findall(r'\w+', s)):
        return None
    return s or None

def decode(text, offsets, probs, thr):
    is_loc = (1 - probs[:, 0]) > thr
    result, start, end = [], None, None
    for t, (t_start, t_end) in enumerate(offsets[:len(probs)]):
        new_loc = is_loc[t] and (probs[t, 1] >= probs[t, 2] or t == 0 or not is_loc[t - 1])
        if new_loc:
            if start is not None:
                result.append(text[start:end])
            start, end = t_start, t_end
        elif is_loc[t] and start is not None:
            end = t_end
        else:
            if start is not None:
                result.append(text[start:end])
            start, end = None, None
    if start is not None:
        result.append(text[start:end])
    if POST_RULES:
        result = [c for c in (clean_span(x, POST_RULES) for x in result) if c]
    return result

def score(probs_list, texts, offsets_list, golds, thr):
    preds = [decode(t, o, p, thr) for t, o, p in zip(texts, offsets_list, probs_list)]
    return comp_metric(golds, preds)

In [5]:
class NERDataset(Dataset):
    def __init__(self, ids, labels):
        self.ids, self.labels = ids, labels
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return torch.tensor(self.ids[i][:MAX_LEN]), torch.tensor(self.labels[i][:MAX_LEN])

def collate(batch):
    ids, labels = zip(*batch)
    return {
        'input_ids': pad_sequence(ids, batch_first=True, padding_value=PAD),
        'labels':    pad_sequence(labels, batch_first=True, padding_value=IGNORE),
        'lengths':   torch.tensor([len(x) for x in ids]),
    }

@torch.no_grad()
def predict_probs(model, ids_list, batch_size=128):
    model.eval()
    chunks = []
    for d, ids in enumerate(ids_list):
        for s in range(0, max(len(ids), 1), MAX_LEN):
            chunks.append((d, s, ids[s:s + MAX_LEN]))
    pieces = {}
    for b in range(0, len(chunks), batch_size):
        batch = chunks[b:b + batch_size]
        x = pad_sequence([torch.tensor(c[2] if c[2] else [PAD]) for c in batch], batch_first=True, padding_value=PAD)
        lengths = torch.tensor([max(len(c[2]), 1) for c in batch])
        probs = torch.softmax(model(x.to(model.device), lengths), dim=-1).cpu().numpy()
        for (d, s, ids), p in zip(batch, probs):
            pieces[(d, s)] = p[:len(ids)]
    out = []
    for d, ids in enumerate(ids_list):
        parts = [pieces[(d, s)] for s in range(0, max(len(ids), 1), MAX_LEN)]
        out.append(np.concatenate(parts) if len(ids) else np.zeros((0, 3), dtype='float32'))
    return out

In [6]:
class NERModel(pl.LightningModule):
    def __init__(self, vocab_size, emb_dim=200, hidden=256, layers=1, bidirectional=True, rnn='lstm',
                 dropout=0.3, lr=2e-3, optimizer='adam', scheduler='onecycle', loc_weight=2.0, emb_matrix=None):
        super().__init__()
        self.save_hyperparameters(ignore=['emb_matrix'])
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        if emb_matrix is not None:
            self.emb.weight.data.copy_(torch.tensor(emb_matrix))
        rnn_cls = nn.LSTM if rnn == 'lstm' else nn.GRU
        self.rnn = rnn_cls(emb_dim, hidden, num_layers=layers, batch_first=True, bidirectional=bidirectional,
                           dropout=dropout if layers > 1 else 0)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hidden * (2 if bidirectional else 1), 3)
        self.register_buffer('class_weight', torch.tensor([1.0, loc_weight, loc_weight]))
        self.val_info = None

    def forward(self, ids, lengths):
        x = self.drop(self.emb(ids))
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=ids.size(1))
        return self.out(self.drop(out))

    def _loss(self, batch):
        logits = self(batch['input_ids'], batch['lengths'])
        return nn.functional.cross_entropy(logits.reshape(-1, 3), batch['labels'].reshape(-1),
                                           weight=self.class_weight, ignore_index=IGNORE)

    def training_step(self, batch, batch_idx):
        loss = self._loss(batch)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        self.log('val_loss', self._loss(batch), prog_bar=True)

    def on_before_optimizer_step(self, optimizer):
        norms = [p.grad.norm() ** 2 for p in self.parameters() if p.grad is not None]
        self.log('grad_norm', torch.sqrt(sum(norms)))

    def on_validation_epoch_end(self):
        if self.val_info is None or self.trainer.sanity_checking:
            return
        texts, offsets, golds, ids = self.val_info
        probs = predict_probs(self, ids)
        self.log('val_f1', score(probs, texts, offsets, golds, thr=0.5), prog_bar=True)

    def configure_optimizers(self):
        lr = self.hparams.lr
        if self.hparams.optimizer == 'adam':
            opt = torch.optim.Adam(self.parameters(), lr=lr)
        elif self.hparams.optimizer == 'adamw':
            opt = torch.optim.AdamW(self.parameters(), lr=lr, weight_decay=0.01)
        else:
            opt = torch.optim.SGD(self.parameters(), lr=lr, momentum=0.9)
        steps = self.trainer.estimated_stepping_batches
        if self.hparams.scheduler == 'onecycle':
            sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
        elif self.hparams.scheduler == 'cosine':
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
        else:
            return opt
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sch, 'interval': 'step'}}

In [7]:
from tokenizers import Tokenizer

STREETS_GEN = ['Шевченка', 'Лесі Українки', 'Миру', 'Незалежності', 'Франка', 'Грушевського', 'Героїв України',
               'Гагаріна', 'Бандери', 'Хрещатик', 'Перемоги', 'Мазепи', 'Івасюка', 'Стуса', 'Сагайдачного']
STREETS_ADJ = ['Центральна', 'Соборна', 'Київська', 'Садова', 'Шкільна', 'Польова', 'Солом\'янська', 'Здолбунівська',
               'Покровська', 'Лісова', 'Набережна', 'Миколаївська']
ABBR = ['вул.', 'просп.', 'пров.', 'бул.']
FULL = ['вулиця', 'проспект', 'провулок', 'бульвар']
CITIES = ['Київ', 'Львів', 'Харків', 'Одеса', 'Дніпро', 'Біла Церква', 'Бровари', 'Ірпінь', 'Буча', 'Вишневе',
          'Фастів', 'Бориспіль', 'Чернігів', 'Вінниця', 'Житомир', 'Полтава']
LOCATIVES = ['Києві', 'Львові', 'Харкові', 'Одесі', 'Дніпрі', 'Броварах', 'Бучі', 'Ірпені', 'Фастові', 'Борисполі',
             'Василькові', 'Вишгороді', 'Обухові', 'Яготині', 'Білій Церкві', 'Чернігові', 'Вінниці', 'Житомирі',
             'Полтаві', 'Миронівці', 'Баришівці', 'Березані', 'Оболоні', 'Борщагівці', 'Позняках', 'Осокорках',
             'Теремках', 'Виноградарі', 'Нивках', 'Деснянському районі', 'Дарницькому районі']
TEMPLATES = [('Адреса: {A}.', True), ('Наша адреса — {A}.', True), ('{A} — гарне місце.', True),
             ('Звертайтесь: {C}, {A}.', True), ('Це сталося на {A}.', False),
             ('Магазин розташований на {A}.', False), ('Ми працюємо в місті {C}, {A}.', False),
             ('Це сталося в {L}.', False), ('Нещодавно в {L} знайшли загублений гаманець.', False)]

def make_address(rng, nominative):
    if rng.random() < 0.4:
        prefix = 'вулиця' if (nominative and rng.random() < 0.5) else 'вул.'
        street = rng.choice(STREETS_ADJ)
    else:
        prefix = rng.choice(FULL if (nominative and rng.random() < 0.5) else ABBR)
        street = rng.choice(STREETS_GEN)
    addr = f'{prefix} {street}'
    if rng.random() < 0.6:
        addr += f', {rng.randint(1, 150)}'
    return addr

def tokenize_words(text):
    toks = list(re.finditer(r"\w+|[^\w\s]", text))
    return [m.group() for m in toks], [(m.start(), m.end()) for m in toks]

def make_labels(offsets, spans):
    labels = [0] * len(offsets)
    for s_start, s_end in spans:
        first = True
        for i, (t_start, t_end) in enumerate(offsets):
            if t_start < s_end and t_end > s_start:
                labels[i] = 1 if first else 2
                first = False
    return labels

def make_synthetic(n, kind, rng):
    word_vocab = prepared['word']['vocab']
    if kind == 'bpe':
        if 'tokenizer_json' in prepared['bpe']:
            bpe = Tokenizer.from_str(prepared['bpe']['tokenizer_json'])
        else:
            bpe = Tokenizer.from_file(find('bpe.json'))
    ids_list, labels_list = [], []
    for _ in range(n):
        tpl, nominative = rng.choice(TEMPLATES)
        addr = make_address(rng, nominative)
        values = {'{A}': addr, '{C}': rng.choice(CITIES), '{L}': rng.choice(LOCATIVES)}
        text, spans = '', []
        for part in re.split(r'(\{[ACL]\})', tpl):
            if part in values:
                spans.append((len(text), len(text) + len(values[part])))
                text += values[part]
            else:
                text += part
        if kind == 'word':
            tokens, offsets = tokenize_words(text)
            ids = [word_vocab.get(w.lower(), UNK) for w in tokens]
        else:
            enc = bpe.encode(text)
            offsets = enc.offsets
            ids = [bpe.token_to_id(t) if bpe.token_to_id(t) is not None else UNK for t in enc.tokens]
        ids_list.append(ids)
        labels_list.append(make_labels(offsets, spans))
    return ids_list, labels_list

demo_ids, demo_labels = make_synthetic(3, 'word', random.Random(SEED))
print(demo_ids[0]); print(demo_labels[0])

[1, 70403, 93, 3228, 789, 14]
[1, 2, 0, 0, 0, 0]


In [8]:
if "results" not in globals():
    results = []

def run_experiment(name, kind='word', rnn='lstm', layers=1, bidirectional=True, hidden=256, emb_dim=200,
                   lr=2e-3, optimizer='adam', scheduler='onecycle', dropout=0.3, loc_weight=2.0,
                   synth=0, epochs=EPOCHS, use_pretrained=False, n_train=None, use_lab=0):
    pl.seed_everything(SEED)
    data = prepared[kind]
    vocab_size = len(prepared['word']['vocab']) if kind == 'word' else prepared['bpe']['vocab_size']
    emb_matrix = prepared['word']['emb_matrix'] if (use_pretrained and kind == 'word') else None
    if emb_matrix is not None:
        emb_dim = emb_matrix.shape[1]

    idx = random.Random(SEED).sample(range(len(data['big']['ids'])), min(n_train or N_TRAIN, len(data['big']['ids'])))
    tr_ids    = [data['big']['ids'][i] for i in idx]
    tr_labels = [data['big']['labels'][i] for i in idx]
    if synth:
        s_ids, s_labels = make_synthetic(synth, kind, random.Random(SEED))
        tr_ids += s_ids; tr_labels += s_labels
    if use_lab:
        tr_ids += data['lab']['ids'] * use_lab
        tr_labels += data['lab']['labels'] * use_lab

    keep = [i for i, x in enumerate(tr_ids) if len(x) > 0]
    tr_ids, tr_labels = [tr_ids[i] for i in keep], [tr_labels[i] for i in keep]

    train_dl = DataLoader(NERDataset(tr_ids, tr_labels), batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=2)
    val_dl   = DataLoader(NERDataset(data['val']['ids'], data['val']['labels']), batch_size=256, collate_fn=collate)

    model = NERModel(vocab_size, emb_dim, hidden, layers, bidirectional, rnn, dropout, lr, optimizer, scheduler,
                     loc_weight, emb_matrix)
    model.val_info = (val_text, data['val']['offsets'], data['val']['locs'], data['val']['ids'])

    loggers = [pl.loggers.CSVLogger('/kaggle/working/logs', name=name)]
    if USE_WANDB:
        wandb_logger = pl.loggers.WandbLogger(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=name)
        loggers.append(wandb_logger)
    ckpt = pl.callbacks.ModelCheckpoint(dirpath=f'/kaggle/working/ckpt/{name}', monitor='val_f1', mode='max', save_top_k=1)
    trainer = pl.Trainer(max_epochs=epochs, accelerator='auto', devices=1, gradient_clip_val=1.0,
                         logger=loggers, callbacks=[ckpt], log_every_n_steps=50, enable_model_summary=False)
    trainer.fit(model, train_dl, val_dl)

    model = NERModel.load_from_checkpoint(ckpt.best_model_path).to('cuda' if torch.cuda.is_available() else 'cpu')
    probs = {'val':  predict_probs(model, data['val']['ids']),
             'lab':  predict_probs(model, data['lab']['ids']),
             'test': predict_probs(model, data['test']['ids'])}

    thrs = np.arange(0.1, 0.91, 0.05)
    val_scores = [score(probs['val'], val_text, data['val']['offsets'], data['val']['locs'], t) for t in thrs]
    thr = float(thrs[int(np.argmax(val_scores))])
    row = {'name': name, 'kind': kind, 'rnn': rnn, 'layers': layers, 'bidir': bidirectional, 'optimizer': optimizer,
           'scheduler': scheduler, 'synth': synth, 'thr': round(thr, 2),
           'val_f1@0.5': score(probs['val'], val_text, data['val']['offsets'], data['val']['locs'], 0.5),
           'val_f1': max(val_scores),
           'lab_f1': None if use_lab else score(probs['lab'], lab_text, data['lab']['offsets'], data['lab']['locs'], thr),
           'lab_in_train': use_lab,
           'LB': None}
    results.append(row)
    with open(f'/kaggle/working/preds_{name}.pkl', 'wb') as f:
        pickle.dump({'probs': probs, 'thr': thr, 'kind': kind}, f)
    if USE_WANDB:
        wandb_logger.experiment.summary.update({k: v for k, v in row.items() if k != 'LB' and v is not None})
        wandb.finish()
    print(row)
    return probs, thr

In [9]:
POST_RULES = {'newline', 'emoji', 'letters', 'upper'}

## Навчання

In [10]:
run_experiment('word_lstm_full', kind='word', rnn='lstm', synth=40_000, n_train=10**9, epochs=4)

Seed set to 21
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=4` reached.


epoch,▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▆▆▆▆▆▆▆▆▆▆▆██████████
grad_norm,█▆▆▅▄▅▃▃▄▄▃▄▂▃▂▃▁▂▂▃▂▂▁▃▂▃▂▃▂▂▂▂▁▃▂▂▂▂▂▁
train_loss,█▇▇▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁
trainer/global_step,▁▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇████
val_f1,▁▆██
val_loss,█▂▁▁
bidir,True
epoch,3
grad_norm,0.03983
kind,word
lab_f1,0.5641


{'name': 'word_lstm_full', 'kind': 'word', 'rnn': 'lstm', 'layers': 1, 'bidir': True, 'optimizer': 'adam', 'scheduler': 'onecycle', 'synth': 40000, 'thr': 0.5, 'val_f1@0.5': 0.84764326, 'val_f1': 0.84764326, 'lab_f1': 0.56410256, 'lab_in_train': 0, 'LB': None}


({'val': [array([[9.99997377e-01, 2.59540968e-07, 2.33463084e-06],
          [9.99961615e-01, 3.71556052e-05, 1.21647076e-06],
          [9.99956608e-01, 3.96481919e-05, 3.64029211e-06],
          [9.97495472e-01, 2.38128100e-03, 1.23215345e-04],
          [9.99619484e-01, 2.66760471e-04, 1.13684473e-04],
          [9.99905348e-01, 6.86544154e-05, 2.59530734e-05],
          [9.99985576e-01, 6.39029167e-06, 8.01061287e-06],
          [9.99966383e-01, 4.83701251e-06, 2.87713283e-05],
          [9.99999166e-01, 4.75222180e-07, 3.96289721e-07],
          [9.99974251e-01, 2.26811717e-05, 3.10704627e-06],
          [9.99993682e-01, 1.02506556e-06, 5.26942722e-06],
          [9.99996424e-01, 2.15577302e-06, 1.48998299e-06],
          [9.99974608e-01, 2.72426178e-06, 2.26467710e-05],
          [9.99988556e-01, 5.91222442e-06, 5.42491716e-06],
          [9.99990106e-01, 5.52260326e-06, 4.35149923e-06],
          [9.99968886e-01, 1.62911947e-05, 1.48166037e-05],
          [9.99875784e-01, 1.1317

In [11]:
run_experiment('word_gru_full', kind='word', rnn='gru', synth=40_000, n_train=10**9, epochs=4)

Seed set to 21
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=4` reached.


epoch,▁▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▃▆▆▆▆▆▆▆▆▆█████████
grad_norm,█▃▂▃▃▂▃▂▂▃▂▂▂▂▂▂▂▂▁▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▂
train_loss,█▇▄▃▃▄▂▂▂▂▂▂▂▁▃▂▂▁▂▁▁▁▁▂▁▁▁▂▁▁▁▁▁▂▂▁▁▁▁▁
trainer/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇███
val_f1,▁▇██
val_loss,█▁▁▁
bidir,True
epoch,3
grad_norm,0.05598
kind,word
lab_f1,0.575


{'name': 'word_gru_full', 'kind': 'word', 'rnn': 'gru', 'layers': 1, 'bidir': True, 'optimizer': 'adam', 'scheduler': 'onecycle', 'synth': 40000, 'thr': 0.45, 'val_f1@0.5': 0.84307079, 'val_f1': 0.8444004, 'lab_f1': 0.575, 'lab_in_train': 0, 'LB': None}


({'val': [array([[9.99947190e-01, 1.20728782e-05, 4.07476764e-05],
          [9.99938369e-01, 4.62856260e-05, 1.54279733e-05],
          [9.99933004e-01, 5.99218401e-05, 6.99423344e-06],
          [9.99383688e-01, 5.95601916e-04, 2.06710192e-05],
          [9.99851704e-01, 1.45502796e-04, 2.79653636e-06],
          [9.99940634e-01, 5.49111755e-05, 4.40949179e-06],
          [9.99983907e-01, 1.05875843e-05, 5.51731591e-06],
          [9.99981642e-01, 1.02542135e-05, 8.07558899e-06],
          [9.99995708e-01, 3.42754174e-06, 8.80670200e-07],
          [9.99990821e-01, 7.02845955e-06, 2.15984187e-06],
          [9.99997735e-01, 5.78705624e-07, 1.72230000e-06],
          [9.99995112e-01, 2.91131005e-06, 2.01037278e-06],
          [9.99997020e-01, 7.06391916e-07, 2.27073656e-06],
          [9.99981761e-01, 1.08695895e-05, 7.35231924e-06],
          [9.99992967e-01, 4.53228449e-06, 2.50587414e-06],
          [9.99979854e-01, 6.95733252e-06, 1.32409796e-05],
          [9.99922633e-01, 6.1840

In [12]:
run_experiment('word_lstm_full_lab', kind='word', rnn='lstm', synth=40_000, n_train=10**9, epochs=4, use_lab=20)

Seed set to 21
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=4` reached.


epoch,▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▃▆▆▆▆▆▆▆▆▆▆██████████
grad_norm,▇▆▇▅█▄▄▅▃▄▄▅█▂▄▃▃▃▄▃▃▂▂▂▂▂▂▂▂▂▂▃▂▁▅▂▂▂▂▂
train_loss,▇▇▆█▇▄▅▄▃▃▂▂▂▂▃▂▂▁▂▂▁▂▂▂▁▂▂▁▁▁▁▂▁▁▁▁▁▁▂▁
trainer/global_step,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇█████
val_f1,▁▆██
val_loss,█▂▁▂
bidir,True
epoch,3
grad_norm,0.03254
kind,word
lab_in_train,20


{'name': 'word_lstm_full_lab', 'kind': 'word', 'rnn': 'lstm', 'layers': 1, 'bidir': True, 'optimizer': 'adam', 'scheduler': 'onecycle', 'synth': 40000, 'thr': 0.4, 'val_f1@0.5': 0.85061096, 'val_f1': 0.85180136, 'lab_f1': None, 'lab_in_train': 20, 'LB': None}


({'val': [array([[9.99993801e-01, 7.15655574e-07, 5.43738724e-06],
          [9.99982238e-01, 1.59396004e-05, 1.74053537e-06],
          [9.99982953e-01, 1.61517928e-05, 9.73275405e-07],
          [9.99927282e-01, 6.22682201e-05, 1.05282852e-05],
          [9.99983072e-01, 1.64378616e-05, 4.90143293e-07],
          [9.99991655e-01, 7.95185588e-06, 3.21427592e-07],
          [9.99998689e-01, 8.25155780e-07, 4.54437497e-07],
          [9.99992371e-01, 2.11199654e-06, 5.50438108e-06],
          [9.99999642e-01, 1.81931142e-07, 1.32485027e-07],
          [9.99995112e-01, 2.64099276e-06, 2.26646148e-06],
          [9.99998331e-01, 6.71065123e-08, 1.55737212e-06],
          [1.00000000e+00, 5.80771626e-08, 1.51420991e-08],
          [9.99999523e-01, 3.45346862e-08, 5.10139273e-07],
          [9.99999285e-01, 5.81434335e-07, 8.71084538e-08],
          [9.99998212e-01, 1.45626564e-06, 3.55628771e-07],
          [9.99997020e-01, 5.75177239e-07, 2.38331381e-06],
          [9.99974132e-01, 2.5279

In [13]:
run_experiment('word_gru_full_lab', kind='word', rnn='gru', synth=40_000, n_train=10**9, epochs=4, use_lab=20)

Seed set to 21
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=4` reached.


epoch,▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▃▃▃▆▆▆▆▆▆▆▆▆█████████
grad_norm,█▅▄▄▃▃▃▃▄▄▂▃▂▂▃▃▅▃▃▂▂▁▂▂▃▂▂▂▁▁▃▃▁▂▃▂▁▂▁▁
train_loss,█▇▅▅▃▃▄▂▂▂▂▂▃▂▂▁▂▂▁▂▁▂▁▁▁▁▂▁▁▁▂▂▂▁▁▁▁▁▁▂
trainer/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
val_f1,▁▇██
val_loss,█▂▁▁
bidir,True
epoch,3
grad_norm,0.07305
kind,word
lab_in_train,20


{'name': 'word_gru_full_lab', 'kind': 'word', 'rnn': 'gru', 'layers': 1, 'bidir': True, 'optimizer': 'adam', 'scheduler': 'onecycle', 'synth': 40000, 'thr': 0.25, 'val_f1@0.5': 0.83406814, 'val_f1': 0.84102762, 'lab_f1': None, 'lab_in_train': 20, 'LB': None}


({'val': [array([[9.99989629e-01, 2.04416347e-06, 8.34663570e-06],
          [9.99985814e-01, 1.26413106e-05, 1.56280851e-06],
          [9.99978781e-01, 2.04543721e-05, 6.59851537e-07],
          [9.99800622e-01, 1.88034130e-04, 1.12674561e-05],
          [9.99941349e-01, 5.66916788e-05, 1.88622073e-06],
          [9.99974251e-01, 2.48826618e-05, 8.35117703e-07],
          [9.99993682e-01, 4.99938142e-06, 1.30533363e-06],
          [9.99993682e-01, 4.02140859e-06, 2.26188922e-06],
          [9.99997735e-01, 1.83726615e-06, 4.77822311e-07],
          [9.99994159e-01, 4.94937740e-06, 7.99228019e-07],
          [9.99999046e-01, 1.13257506e-07, 8.82206393e-07],
          [9.99999285e-01, 6.32510023e-07, 9.18274452e-08],
          [9.99999642e-01, 9.81653727e-08, 2.31867219e-07],
          [9.99998689e-01, 1.19821311e-06, 1.63118429e-07],
          [9.99998808e-01, 1.18797402e-06, 4.95568244e-08],
          [9.99998450e-01, 1.24808014e-06, 3.81454385e-07],
          [9.99994397e-01, 5.2411

## Результати

In [14]:
res = pd.DataFrame(results)
res.sort_values('val_f1', ascending=False)

,name,kind,rnn,layers,bidir,optimizer,scheduler,synth,thr,val_f1@0.5,val_f1,lab_f1,lab_in_train,LB
2,word_lstm_full_lab,word,lstm,1,True,adam,onecycle,40000,0.40,0.850611,0.851801,NaN,20,None
0,word_lstm_full,word,lstm,1,True,adam,onecycle,40000,0.50,0.847643,0.847643,0.564103,0,None
1,word_gru_full,word,gru,1,True,adam,onecycle,40000,0.45,0.843071,0.844400,0.575000,0,None
3,word_gru_full_lab,word,gru,1,True,adam,onecycle,40000,0.25,0.834068,0.841028,NaN,20,None


## Сабміт

In [15]:
def make_submission(name, path='/kaggle/working/submission.csv'):
    with open(f'/kaggle/working/preds_{name}.pkl', 'rb') as f:
        p = pickle.load(f)
    offsets = prepared[p['kind']]['test']['offsets']
    preds = [decode(t, o, pr, p['thr']) for t, o, pr in zip(test_text, offsets, p['probs']['test'])]
    sub = pd.DataFrame({'text_id': prepared['word']['test'].get('text_id', range(len(test_text))),
                        'locations': [json.dumps(x, ensure_ascii=False) for x in preds]})
    sub.to_csv(path, index=False)
    print(sub.head(), '\nрядків:', len(sub), '| без локацій:', (sub['locations'] == '[]').mean().round(2))